# 01 · Bronze layer: raw ingestion


**Purpose.** Load the source CSV with Spark and keep a raw copy as a Delta table.

**Inputs.** `/Volumes/workspace/car_insurance/raw/Car_Insurance_Claim.csv` (path in `CLAIMS_CSV`).

**Outputs.** `workspace.car_insurance.claims_bronze` (name in `BRONZE_TABLE`).

**Assumptions.** 
- Source file for data is CSV 
- The first row of the CSV file is a header

**What the next notebook relies on (the table contract)**

- All 19 source columns with their original names (`SOURCE_COLUMNS`) and original values.
- One row per record in the source file: 10,000. Malformed records are kept and flagged, not dropped.
- Extra metadata columns (source file, ingestion time) are fine; start their names with `_`.

**Shared helpers.** The first code cell runs `00_setup`, which defines the table names (`BRONZE_TABLE`, `SILVER_TABLE`, ...), the file paths (`CLAIMS_CSV`), and `write_delta` / `validate_tables`. Use those names instead of typing table names, so everyone writes to the same place.

In [0]:
%run ./00_setup

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

require_source_files(CLAIMS_CSV)

## 1. Read the source data
- Malformed rows are marked as "corrupted records" and kept within the Bronze Layer

In [0]:
csv_schema = StructType(
    [StructField(name, StringType(), nullable=True) for name in SOURCE_COLUMNS]  
    + [StructField("_corrupt_record", StringType(), nullable=True)]
)

# read csv 
reader = spark.read
# use schema defined earlier
reader = reader.schema(csv_schema)
# treat first rows as  column names
reader = reader.option("header", "true")
# check the csv headers and column names
reader = reader.option("enforceSchema", "false")
# continue even if data is garbage
reader = reader.option("mode", "PERMISSIVE")
# keep track of parsing issues for CSV rows
reader = reader.option("columnNameOfCorruptRecord", "_corrupt_record")

raw_dataframe = reader.csv(CLAIMS_CSV)
raw_dataframe.printSchema()
display(raw_dataframe)

In [0]:
display(
    raw_dataframe.agg(
        *[
            F.count(F.when(F.col(name).isNull(), 1)).alias(name)
            for name in SOURCE_COLUMNS
        ]
    )
)

In [0]:
bronze_df = raw_dataframe.withColumn("_source_file", F.col("_metadata.file_name"))
bronze_df = bronze_df.withColumn("_ingested_at", F.current_timestamp())

## 2. Write and verify the Delta table

In [0]:
write_delta(bronze_df, BRONZE_TABLE)

# Reads the table back from the catalog and checks format, row count and columns.
validate_tables({BRONZE_TABLE: 10000})